In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error


In [2]:
df = pd.read_csv("../../data/FRB_H15.csv").dropna()

#rename columns
df.rename(columns={"Series Description": "Date", "Market yield on U.S. Treasury securities at 1-month  constant maturity, quoted on investment basis": "0Y1M", "Market yield on U.S. Treasury securities at 3-month  constant maturity, quoted on investment basis": "0Y3M", "Market yield on U.S. Treasury securities at 6-month  constant maturity, quoted on investment basis": "0Y6M", "Market yield on U.S. Treasury securities at 1-year  constant maturity, quoted on investment basis": "1Y", "Market yield on U.S. Treasury securities at 2-year  constant maturity, quoted on investment basis": "2Y", "Market yield on U.S. Treasury securities at 3-year  constant maturity, quoted on investment basis": "3Y", "Market yield on U.S. Treasury securities at 5-year  constant maturity, quoted on investment basis": "5Y", "Market yield on U.S. Treasury securities at 7-year  constant maturity, quoted on investment basis": "7Y", "Market yield on U.S. Treasury securities at 10-year  constant maturity, quoted on investment basis": "10Y", "Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis": "20Y", "Market yield on U.S. Treasury securities at 30-year  constant maturity, quoted on investment basis": "30Y"}, inplace=True)
df.rename(columns={"Market yield on U.S. Treasury securities at 20-year  constant maturity, quoted on investment basis": "20Y"}, inplace=True)

#make index to datetime for timeseries
#df["Date"] = pd.to_datetime(df["Date"])
# dont make timeseries data for xgboost

#df.set_index("Date", inplace=True)
df = df.reset_index(drop=True)
df.head(6)

,Date,0Y1M,0Y3M,0Y6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
0,2001-07-31,3.67,3.54,3.47,3.53,3.79,4.06,4.57,4.86,5.07,5.61,5.51
1,2001-08-01,3.65,3.53,3.47,3.56,3.83,4.09,4.62,4.90,5.11,5.63,5.53
2,2001-08-02,3.65,3.53,3.46,3.57,3.89,4.17,4.69,4.97,5.17,5.68,5.57
3,2001-08-03,3.63,3.52,3.47,3.57,3.91,4.22,4.72,4.99,5.20,5.70,5.59
4,2001-08-06,3.62,3.52,3.47,3.56,3.88,4.17,4.71,4.99,5.19,5.70,5.59
5,2001-08-07,3.63,3.52,3.47,3.56,3.90,4.19,4.72,5.00,5.20,5.71,5.60


In [3]:
#seperate into different dataframes
maturities = []
matList = ["0Y1M", "0Y3M", "0Y6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]

for col in df.columns:
    maturities.append(df[col])

#now create a table for each maturity that has 5 columns, the lag values for 1-20 days

tables = []
features = ['lag1', 'lag2', 'lag3', 'lag4', 'lag5']
# one table for each maturity
for mat in matList:
    
    # table 5 maturities, 
    # ,'6','7','8','9','10', '11','12','13','14','15', '16','17','18','19','20', '21'

    #rows 3 is 1 day forecast, rows2 = 5 day, rows3 = 20 day
    rows1 = []
    rows2 = []
    rows3 = []
    
    for i in range(4, len(df.index) - 5):
            #adds in all 20 lag values for each maturity
        lag1 = df[mat].iloc[i]
        lag2 = df[mat].iloc[i-1]
        lag3 = df[mat].iloc[i-2]
        lag4 = df[mat].iloc[i-3]
        lag5 = df[mat].iloc[i-4]

        #target is value 5 days in advance, so this model is 5 day horizon
        target = df[mat].iloc[i+5]
        rows1.append([lag1, lag2, lag3, lag4, lag5, target])
    for i in range(4, len(df.index) - 20):
        lag1 = df[mat].iloc[i]
        lag2 = df[mat].iloc[i-1]
        lag3 = df[mat].iloc[i-2]
        lag4 = df[mat].iloc[i-3]
        lag5 = df[mat].iloc[i-4]

        target = df[mat].iloc[i+20]
        rows2.append([lag1, lag2, lag3, lag4, lag5, target])
    for i in range(4, len(df.index) - 1):
        lag1 = df[mat].iloc[i]
        lag2 = df[mat].iloc[i-1]
        lag3 = df[mat].iloc[i-2]
        lag4 = df[mat].iloc[i-3]
        lag5 = df[mat].iloc[i-4]

        target = df[mat].iloc[i+1]
        rows3.append([lag1, lag2, lag3, lag4, lag5, target])

    df1 = pd.DataFrame(rows3, columns=["lag1", "lag2", "lag3", "lag4", "lag5", "target"])      
    df2 = pd.DataFrame(rows2, columns=["lag1", "lag2", "lag3", "lag4", "lag5", "target"])
    df3 = pd.DataFrame(rows1, columns=["lag1", "lag2", "lag3", "lag4", "lag5", "target"])      

    #add 1,5,20 day data
    tables.append([df1,df2, df3])


In [4]:
xy = []

for i in range(len(tables)):
    x1 = tables[i][0][features]
    y1 = tables[i][0]["target"]

    x2 = tables[i][1][features]
    y2 = tables[i][1]["target"]

    x3 = tables[i][2][features]
    y3 = tables[i][2]["target"]
    
    xy.append([(x1, y1), (x2, y2), (x3, y3)])

x_train = [[] for i in range(len(matList))]
y_train = [[] for i in range(len(matList))]
x_test = [[] for i in range(len(matList))]
y_test = [[] for i in range(len(matList))]

for i in range(len(matList)):
    # xy[i] is a list of 3 tuples, each tuple is (x,y) for 1,5,20 day forecast
    for j in range(3):
        x,y = xy[i][j]
        split = int(len(x)*0.8)
        x_train[i].append(x[:split])
        y_train[i].append(y[:split])
        x_test[i].append(x[split:])
        y_test[i].append(y[split:])


#models is a list of 3 lists, first index is maturity, second index is 1,5,20 day forecast
models = [[] for i in range(len(matList))]

for i in range(len(x_train)):
    for j in range(3):
        model = xgb.XGBRegressor(
            objective='reg:squarederror',
            n_estimators = 500,
            learning_rate = 0.02,
            max_depth = 3,
            subsample = 0.8,
            colsample_bytree = 0.8, 
            reg_lambda = 5, 
            random_state = 7
        )
        model.fit(x_train[i][j], y_train[i][j])
        models[i].append(model)

predictions = [[] for i in range(len(matList))]
for i in range(len(models)):
    pred = []
    for j in range(3):
        pred.append(models[i][j].predict(x_test[i][j]))
    predictions[i] = pred

#rmse is in basis points
rmse = [[] for i in range(len(matList))]
for i in range(len(predictions)):
    for j in range(3):
        rmse[i].append(root_mean_squared_error(y_test[i][j], predictions[i][j])*100)

for i in range(len(matList)):
    print(f"  Maturity: {matList[i]}")
    for j in range(3):
        if j == 0:
            print(f"  Forecast {j+1} day, RMSE: {rmse[i][j]}")
        elif j == 1:
            print(f"  Forecast {5} day, RMSE: {rmse[i][j]}")
        else:
            print(f"  Forecast {20} day, RMSE: {rmse[i][j]}")



  Maturity: 0Y1M
  Forecast 1 day, RMSE: 18.287978671021264
  Forecast 5 day, RMSE: 42.36348087026986
  Forecast 20 day, RMSE: 26.31388558384961
  Maturity: 0Y3M
  Forecast 1 day, RMSE: 17.757049116576308
  Forecast 5 day, RMSE: 33.44904864837417
  Forecast 20 day, RMSE: 21.366971230666888
  Maturity: 0Y6M
  Forecast 1 day, RMSE: 10.2129491292645
  Forecast 5 day, RMSE: 28.40444628389463
  Forecast 20 day, RMSE: 14.40845622105307
  Maturity: 1Y
  Forecast 1 day, RMSE: 7.63036841619308
  Forecast 5 day, RMSE: 31.013800350346525
  Forecast 20 day, RMSE: 14.045235820717517
  Maturity: 2Y
  Forecast 1 day, RMSE: 7.489559256161395
  Forecast 5 day, RMSE: 32.18065083871436
  Forecast 20 day, RMSE: 15.372563174668002
  Maturity: 3Y
  Forecast 1 day, RMSE: 7.3825994959041585
  Forecast 5 day, RMSE: 32.85960323191844
  Forecast 20 day, RMSE: 15.29158240583423
  Maturity: 5Y
  Forecast 1 day, RMSE: 7.139273844968996
  Forecast 5 day, RMSE: 30.98029047621974
  Forecast 20 day, RMSE: 15.0299321402